# Exp 5 — Majority-Class Downsampling

**Hypothesis:** Capping Grade 0 at Grade 2 count isolates the effect of data-level
rebalancing vs loss-level rebalancing (Exp 1). Uses plain CrossEntropyLoss.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve()))  # so 'from utils import *' works
from utils import *
print("utils loaded ✓")

In [ ]:
EXP_DIR  = ROOT / 'outputs' / 'swinv2_small_exp5'
CKPT_DIR_E = EXP_DIR / 'checkpoints'
FIG_DIR_E  = EXP_DIR / 'figures'
for d in (CKPT_DIR_E, FIG_DIR_E): d.mkdir(parents=True, exist_ok=True)
CKPT_PATH  = CKPT_DIR_E / 'exp5_best.pt'

In [ ]:
# ── Downsample Grade 0 to Grade 2 count ──────────────────────────────────────
grade2_count  = (train_df['level'] == 2).sum()
grade0_capped = train_df[train_df['level'] == 0].sample(n=grade2_count, random_state=SEED)
exp5_train_df = (pd.concat([grade0_capped, train_df[train_df['level'] != 0]])
                 .sample(frac=1, random_state=SEED).reset_index(drop=True))
print(f'Original: {len(train_df):,} | Exp5: {len(exp5_train_df):,}')
print()
for g, name in enumerate(CLASS_NAMES):
    orig = (train_df['level']==g).sum()
    new  = (exp5_train_df['level']==g).sum()
    note = '  ← capped' if g==0 else ''
    print(f'  {g} {name:<18}: {orig:>6,} → {new:>6,}{note}')

In [ ]:
train_ds    = DRDataset(exp5_train_df, TRAIN_DIR, transform=baseline_train_transform)
val_ds      = DRDataset(val_df,        VAL_DIR,   transform=val_transform)
test_ds     = DRDataset(test_df,       TEST_LABELS_DIR, transform=val_transform)
t_loader    = make_loader(train_ds)   # no sampler — downsampling provides balance
v_loader    = make_val_loader(val_ds)
test_loader = make_val_loader(test_ds)
print('No WeightedRandomSampler — downsampling alone provides class balance.')

In [ ]:
set_seed()
model     = build_swin_model(pretrained=True).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)   # plain CE — no class weights
optimizer = layer_wise_optimizer(model)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = torch.amp.GradScaler('cuda', enabled=use_amp)
print('Loss: plain CrossEntropyLoss (no class weights) — data-level balance only.')

In [ ]:
history, best_kappa = train_loop(
    model, criterion, optimizer, scheduler, scaler,
    t_loader, v_loader, CKPT_PATH)

In [ ]:
y_true, y_pred = evaluate(model, criterion, optimizer, scaler,
                           test_loader, CKPT_PATH, 'Exp5')
plot_confusion(y_true, y_pred, 'Exp5 Downsample', FIG_DIR_E / 'exp5_confusion.png', cmap='YlOrRd')

import json as _json
exp5_qwk = cohen_kappa_score(y_true, y_pred, weights='quadratic')
results = {'qwk': exp5_qwk, 'macro_f1': f1_score(y_true, y_pred, average='macro'),
           'f1_per': f1_score(y_true, y_pred, average=None).tolist(), 'history': history}
with open(EXP_DIR / 'exp5_results.json', 'w') as f:
    _json.dump(results, f, indent=2)

# Key insight: data-level vs loss-level rebalancing
try:
    with open(ROOT/'outputs'/'swinv2_small_exp1'/'exp1_results.json') as f:
        e1 = _json.load(f)
    print(f'\nData-level (Exp5) QWK:  {exp5_qwk:.4f}')
    print(f'Loss-level (Exp1) QWK:  {e1["qwk"]:.4f}')
    winner = 'Exp5 (downsampling)' if exp5_qwk > e1['qwk'] else 'Exp1 (WRS+FocalLoss)'
    print(f'Winner: {winner} — discuss why in report.')
except FileNotFoundError:
    print(f'Exp5 QWK: {exp5_qwk:.4f}')